# Nifty & Sensex — Monday-to-Friday Weekly Moves

Measure each calendar week's move from **Monday's 09:15 candle open** to the
**close of Friday's 15:15 candle** (all timestamps are in Asia/Kolkata).
The closing value is the last hourly candle close supplied in the JSON, not a
separately sourced official daily closing value.

| Measure | Calculation |
|---|---|
| Move (points) | Friday close − Monday open |
| Move (%) | 100 × (Friday close − Monday open) / Monday open |
| Max up / down (%) | Highest high / lowest low relative to Monday open |
| Range (%) | 100 × (highest high − lowest low) / Monday open |

Weeks use calendar Mondays and Fridays, independently of option expiry dates.
Missing Monday/Friday endpoint candles are **never replaced by another day**.
Weekend and nonstandard session candles are excluded. All seven expected hourly
candles (09:15 through 15:15) must exist on each observed session for a week to
enter the analysis. Weeks with a missing middle weekday remain eligible by default,
but are flagged; set `REQUIRE_FIVE_SESSIONS = True` to require all five weekdays.
Without an exchange calendar, a missing weekday cannot be classified as a holiday
versus a data gap. High/low ranges only cover the supplied candles.

The coverage report uses the actual timestamps in the two input files; their
filenames do not guarantee the dates contained within them.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display


def find_project_root():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "data/raw/option_selling").is_dir():
            return candidate
    raise FileNotFoundError("Run from the project root or a notebook subfolder.")


PROJECT_ROOT = find_project_root()
SOURCES = {
    "Nifty 50": PROJECT_ROOT / "data/raw/option_selling/nifty/nifty50_hourly_2020-01-01_2026-11-06.json",
    "Sensex": PROJECT_ROOT / "data/raw/option_selling/sensex/sensex_hourly_2020-01-01_2026-11-06.json",
}
REQUIRE_FIVE_SESSIONS = False
START_WEEK = None  # Optional Monday, e.g. "2024-01-01"; inclusive.
END_WEEK = None    # Optional Monday; inclusive. Filters whole weeks.
EXPECTED_TIMES = tuple(f"{hour:02d}:15" for hour in range(9, 16))
COLORS = {"Nifty 50": "#2a78d6", "Sensex": "#eb6834"}
EXPORT_DIR = PROJECT_ROOT / "data/processed/option_selling/weekly_monday_friday"

pio.templates.default = "plotly_white"
# Store interactive figure data and an HTML fallback in the notebook.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

## Load and check the candles

Exact duplicate records are removed and reported; conflicting records for the
same timestamp raise an error. Prices must be finite, positive and consistent
with their OHLC bounds. Source coverage and excluded candles are shown below.

In [2]:
def load_candles(path):
    with path.open() as handle:
        rows = json.load(handle)["data"]["candles"]
    if not rows or any(len(row) != 7 for row in rows):
        raise ValueError(f"{path.name}: expected nonempty seven-field OHLC candles")
    raw = pd.DataFrame(rows, columns=["timestamp", "open", "high", "low", "close", "volume", "oi"])
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True, errors="raise").dt.tz_convert("Asia/Kolkata")
    if raw["timestamp"].isna().any():
        raise ValueError(f"{path.name}: missing timestamps")
    source_rows = len(raw)
    raw = raw.drop_duplicates()
    duplicates_removed = source_rows - len(raw)
    if raw["timestamp"].duplicated().any():
        raise ValueError(f"{path.name}: conflicting duplicate timestamps")
    raw = raw.set_index("timestamp").sort_index()
    ohlc = raw[["open", "high", "low", "close"]].apply(pd.to_numeric, errors="raise")
    valid = (
        np.isfinite(ohlc).all(axis=1)
        & ohlc.gt(0).all(axis=1)
        & ohlc["high"].ge(ohlc[["open", "close", "low"]].max(axis=1))
        & ohlc["low"].le(ohlc[["open", "close", "high"]].min(axis=1))
    )
    if not valid.all():
        raise ValueError(f"{path.name}: {int((~valid).sum())} invalid OHLC records")
    weekday = ohlc.index.dayofweek < 5
    regular_time = ohlc.index.strftime("%H:%M:%S").isin([t + ":00" for t in EXPECTED_TIMES])
    price = ohlc.loc[weekday & regular_time].copy()
    if price.empty:
        raise ValueError(f"{path.name}: no regular weekday candles")
    # Calendar grouping below uses local dates; retain the original timezone above.
    price.index = price.index.tz_localize(None)
    price["session"] = price.index.normalize()
    price["week_start"] = price["session"] - pd.to_timedelta(price.index.dayofweek, unit="D")
    audit = {
        "source_file": path.name,
        "first_source_candle_ist": raw.index.min().isoformat(),
        "last_source_candle_ist": raw.index.max().isoformat(),
        "source_rows": source_rows,
        "exact_duplicates_removed": duplicates_removed,
        "weekend_candles_excluded": int((~weekday).sum()),
        "nonstandard_weekday_candles_excluded": int((weekday & ~regular_time).sum()),
        "regular_candles": len(price),
        "observed_sessions": price["session"].nunique(),
    }
    return price, audit


prices, coverage_rows = {}, []
for instrument, path in SOURCES.items():
    prices[instrument], audit = load_candles(path)
    coverage_rows.append({"instrument": instrument, **audit})
coverage = pd.DataFrame(coverage_rows).set_index("instrument")
display(coverage.T)

instrument,Nifty 50,Sensex
source_file,nifty50_2020-01-01_2026-09-15.json,sensex_hourly_2020-01-01_2026-09-19.json
first_source_candle_ist,2022-01-03T09:15:00+05:30,2022-01-03T09:15:00+05:30
last_source_candle_ist,2026-09-15T15:15:00+05:30,2026-09-18T15:15:00+05:30
source_rows,8130,8152
exact_duplicates_removed,0,0
weekend_candles_excluded,28,28
nonstandard_weekday_candles_excluded,3,3
regular_candles,8099,8121
observed_sessions,1157,1161


## Build calendar weeks

`endpoint_move_pct` preserves the return whenever both endpoint candles exist.
The tables, statistics and charts below use only `included` weeks: both endpoints,
complete hourly coverage on observed sessions, and the selected five-session rule.
Missing dates and exclusion reasons stay in the audit and CSV exports. Entirely
missing weeks inside each file's regular-session coverage also appear in the audit.

In [3]:
def build_weekly_moves(price, instrument, require_five_sessions=False):
    rows = []
    week_groups = dict(tuple(price.groupby("week_start")))
    for monday in pd.date_range(price["week_start"].min(), price["week_start"].max(), freq="W-MON"):
        friday = monday + pd.Timedelta(days=4)
        window = week_groups.get(monday, price.iloc[:0])
        monday_ts = monday + pd.Timedelta(hours=9, minutes=15)
        friday_ts = friday + pd.Timedelta(hours=15, minutes=15)
        has_monday, has_friday = monday_ts in window.index, friday_ts in window.index
        observed = pd.DatetimeIndex(window["session"].unique())
        missing_dates = pd.date_range(monday, friday, freq="D").difference(observed)
        incomplete = []
        for session, candles in window.groupby("session"):
            if tuple(candles.index.strftime("%H:%M")) != EXPECTED_TIMES:
                incomplete.append(session.strftime("%Y-%m-%d"))
        issues = []
        if not has_monday:
            issues.append("Missing Monday 09:15 open")
        if not has_friday:
            issues.append("Missing Friday 15:15 close")
        if incomplete:
            issues.append("Incomplete hourly session(s)")
        if require_five_sessions and len(observed) != 5:
            issues.append("Fewer than five sessions")
        anchor = window.at[monday_ts, "open"] if has_monday else np.nan
        close = window.at[friday_ts, "close"] if has_friday else np.nan
        high = window["high"].max()
        low = window["low"].min()
        rows.append({
            "instrument": instrument,
            "week_start": monday,
            "week_end": friday,
            "first_observed_candle": window.index.min(),
            "last_observed_candle": window.index.max(),
            "trading_days": len(observed),
            "candles": len(window),
            "missing_weekdays": ", ".join(missing_dates.strftime("%Y-%m-%d (%a)")),
            "incomplete_sessions": ", ".join(incomplete),
            "has_both_endpoints": has_monday and has_friday,
            "full_five_sessions": len(observed) == 5 and not incomplete,
            "included": not issues,
            "status": "; ".join(issues) if issues else (
                "Included: 5 sessions" if len(observed) == 5 else "Included: missing middle weekday(s)"
            ),
            "monday_open": anchor,
            "friday_close": close,
            "endpoint_move_points": close - anchor,
            "endpoint_move_pct": (close - anchor) / anchor * 100,
            "weekly_high": high,
            "weekly_low": low,
            "max_up_pct": (high - anchor) / anchor * 100,
            "max_down_pct": (low - anchor) / anchor * 100,
            "range_pct": (high - low) / anchor * 100,
        })
    return pd.DataFrame(rows)


weekly_all = pd.concat([
    build_weekly_moves(price, instrument, REQUIRE_FIVE_SESSIONS)
    for instrument, price in prices.items()
], ignore_index=True)
for bound in [START_WEEK, END_WEEK]:
    if bound is not None and pd.Timestamp(bound).dayofweek != 0:
        raise ValueError("START_WEEK and END_WEEK must be Mondays")
if START_WEEK is not None and END_WEEK is not None and pd.Timestamp(START_WEEK) > pd.Timestamp(END_WEEK):
    raise ValueError("START_WEEK must not be later than END_WEEK")
if START_WEEK is not None:
    weekly_all = weekly_all.loc[weekly_all["week_start"] >= pd.Timestamp(START_WEEK)]
if END_WEEK is not None:
    weekly_all = weekly_all.loc[weekly_all["week_start"] <= pd.Timestamp(END_WEEK)]
weekly = weekly_all.loc[weekly_all["included"]].copy()
weekly["abs_move_pct"] = weekly["endpoint_move_pct"].abs()
excluded = weekly_all.loc[~weekly_all["included"]].copy()

week_counts = weekly_all.groupby("instrument").agg(
    calendar_weeks=("week_start", "size"),
    weeks_with_both_endpoints=("has_both_endpoints", "sum"),
    full_five_session_weeks=("full_five_sessions", "sum"),
    included_weeks=("included", "sum"),
)
week_counts["excluded_weeks"] = week_counts["calendar_weeks"] - week_counts["included_weeks"]
display(week_counts)
if weekly.empty:
    raise ValueError("No eligible Monday-to-Friday weeks for these settings")

,calendar_weeks,weeks_with_both_endpoints,full_five_session_weeks,included_weeks,excluded_weeks
instrument,,,,,
Nifty 50,246,220,184,220,26
Sensex,246,220,184,219,27


## Every week's move

Scroll each table to inspect the full history; newest weeks appear first.
All percentages use Monday's open as the denominator. The last section exports
the complete numeric tables, including the audit, to CSV.

In [4]:
def scroll_table(frame, height=460):
    display(HTML(
        f'<div style="max-height:{height}px;overflow:auto;border:1px solid #ddd">'
        + frame.to_html(index=False, float_format=lambda value: f"{value:,.2f}", na_rep="—")
        + '</div>'
    ))


table_columns = {
    "week_start": "Monday", "week_end": "Friday", "trading_days": "Sessions",
    "monday_open": "Monday open", "friday_close": "Friday close",
    "endpoint_move_points": "Move (points)", "endpoint_move_pct": "Move (%)",
    "max_up_pct": "Max up (%)", "max_down_pct": "Max down (%)", "range_pct": "Range (%)",
    "missing_weekdays": "Missing weekdays",
}
for instrument in SOURCES:
    display(HTML(f"<h3>{instrument}</h3>"))
    table = weekly.loc[weekly["instrument"].eq(instrument)].sort_values("week_start", ascending=False)
    scroll_table(table[list(table_columns)].rename(columns=table_columns))

display(HTML("<h3>Excluded weeks — endpoints or session coverage missing</h3>"))
scroll_table(excluded[[
    "instrument", "week_start", "week_end", "trading_days", "status",
    "missing_weekdays", "incomplete_sessions",
]].sort_values(["week_start", "instrument"], ascending=[False, True]), height=320)

Monday,Friday,Sessions,Monday open,Friday close,Move (points),Move (%),Max up (%),Max down (%),Range (%),Missing weekdays
2026-09-07,2026-09-11,5,"23,883.15","23,398.10",-485.05,-2.03,0.03,-2.73,2.76,
2026-08-31,2026-09-04,5,"24,117.55","23,897.70",-219.85,-0.91,0.11,-1.37,1.48,
2026-08-24,2026-08-28,5,"24,285.05","24,175.65",-109.40,-0.45,0.39,-0.86,1.24,
2026-08-17,2026-08-21,5,"24,343.45","24,252.00",-91.45,-0.38,0.07,-1.31,1.37,
2026-08-10,2026-08-14,5,"24,581.25","24,366.00",-215.25,-0.88,0.16,-1.28,1.44,
2026-08-03,2026-08-07,5,"24,572.70","24,570.65",-2.05,-0.01,0.82,-0.59,1.41,
2026-07-27,2026-07-31,5,"23,928.40","24,366.70",438.30,1.83,2.09,-0.15,2.25,
2026-07-20,2026-07-24,5,"24,190.05","23,787.00",-403.05,-1.67,0.31,-2.41,2.73,
2026-07-13,2026-07-17,5,"24,039.40","24,346.70",307.30,1.28,1.36,-0.16,1.53,
2026-07-06,2026-07-10,5,"24,306.85","24,211.65",-95.20,-0.39,0.92,-2.06,2.99,


Monday,Friday,Sessions,Monday open,Friday close,Move (points),Move (%),Max up (%),Max down (%),Range (%),Missing weekdays
2026-09-07,2026-09-11,5,"76,446.05","74,781.76","-1,664.29",-2.18,0.04,-2.99,3.03,
2026-08-31,2026-09-04,5,"77,130.73","76,515.43",-615.30,-0.80,0.13,-1.29,1.42,
2026-08-24,2026-08-28,5,"77,629.56","77,264.51",-365.05,-0.47,0.46,-0.90,1.36,
2026-08-17,2026-08-21,5,"77,892.92","77,540.83",-352.09,-0.45,0.05,-1.37,1.42,
2026-08-10,2026-08-14,5,"78,501.59","78,009.25",-492.34,-0.63,0.22,-1.28,1.50,
2026-08-03,2026-08-07,5,"78,883.34","78,499.17",-384.17,-0.49,0.33,-0.85,1.18,
2026-07-27,2026-07-31,5,"76,608.98","78,034.57","1,425.59",1.86,2.17,-0.12,2.29,
2026-07-20,2026-07-24,5,"78,151.45","76,141.27","-2,010.18",-2.57,0.00,-3.43,3.43,
2026-07-13,2026-07-17,5,"76,963.35","78,256.34","1,292.99",1.68,1.71,-0.14,1.85,
2026-07-06,2026-07-10,5,"77,940.90","77,593.11",-347.79,-0.45,0.93,-2.16,3.09,


instrument,week_start,week_end,trading_days,status,missing_weekdays,incomplete_sessions
Nifty 50,2026-09-14,2026-09-18,1,Missing Monday 09:15 open; Missing Friday 15:15 close,"2026-09-14 (Mon), 2026-09-16 (Wed), 2026-09-17 (Thu), 2026-09-18 (Fri)",
Sensex,2026-09-14,2026-09-18,4,Missing Monday 09:15 open,2026-09-14 (Mon),
Nifty 50,2026-06-22,2026-06-26,4,Missing Friday 15:15 close,2026-06-26 (Fri),
Sensex,2026-06-22,2026-06-26,4,Missing Friday 15:15 close,2026-06-26 (Fri),
Nifty 50,2026-04-27,2026-05-01,4,Missing Friday 15:15 close,2026-05-01 (Fri),
Sensex,2026-04-27,2026-05-01,4,Missing Friday 15:15 close,2026-05-01 (Fri),
Nifty 50,2026-03-30,2026-04-03,3,Missing Friday 15:15 close,"2026-03-31 (Tue), 2026-04-03 (Fri)",
Sensex,2026-03-30,2026-04-03,3,Missing Friday 15:15 close,"2026-03-31 (Tue), 2026-04-03 (Fri)",
Nifty 50,2026-01-26,2026-01-30,4,Missing Monday 09:15 open,2026-01-26 (Mon),
Sensex,2026-01-26,2026-01-30,4,Missing Monday 09:15 open,2026-01-26 (Mon),


## Typical move and extremes

The signed average can cancel gains against losses; the **mean absolute move**
shows typical move size. The 90th percentile is the historical absolute return
at or below which 90% of the sample lies. Counts and date coverage are specific
to each index; the comparison chart uses only weeks included for both indices.

In [5]:
def summarize(group):
    move = group["endpoint_move_pct"]
    return pd.Series({
        "weeks": len(group),
        "up_weeks": int(move.gt(0).sum()),
        "down_weeks": int(move.lt(0).sum()),
        "flat_weeks": int(move.eq(0).sum()),
        "up_weeks_pct": move.gt(0).mean() * 100,
        "mean_move_points": group["endpoint_move_points"].mean(),
        "mean_move_pct": move.mean(),
        "median_move_pct": move.median(),
        "mean_abs_move_pct": move.abs().mean(),
        "p90_abs_move_pct": move.abs().quantile(0.90),
        "best_move_pct": move.max(),
        "worst_move_pct": move.min(),
        "mean_max_up_pct": group["max_up_pct"].mean(),
        "mean_max_down_pct": group["max_down_pct"].mean(),
        "mean_range_pct": group["range_pct"].mean(),
    })


summary = pd.DataFrame({name: summarize(group) for name, group in weekly.groupby("instrument")}).T
summary.index.name = "instrument"
display(summary.T)

extremes = pd.concat([
    group.loc[[group["endpoint_move_pct"].idxmax(), group["endpoint_move_pct"].idxmin()]].assign(
        extreme=["Largest gain", "Largest loss"]
    )
    for _, group in weekly.groupby("instrument")
])
display(extremes[["instrument", "extreme", "week_start", "week_end", "endpoint_move_points", "endpoint_move_pct"]])

annual = pd.DataFrame([
    {"instrument": instrument, "year": year, **summarize(group).to_dict()}
    for (instrument, year), group in weekly.groupby(["instrument", weekly["week_start"].dt.year])
])
display(annual[["instrument", "year", "weeks", "up_weeks_pct", "mean_move_pct", "mean_abs_move_pct", "p90_abs_move_pct"]])

instrument,Nifty 50,Sensex
weeks,220.00,219.00
up_weeks,112.00,113.00
down_weeks,108.00,106.00
flat_weeks,0.00,0.00
up_weeks_pct,50.91,51.60
mean_move_points,18.82,76.43
mean_move_pct,0.11,0.13
median_move_pct,0.07,0.03
mean_abs_move_pct,1.34,1.38
p90_abs_move_pct,2.75,2.75


,instrument,extreme,week_start,week_end,endpoint_move_points,endpoint_move_pct
222,Nifty 50,Largest gain,2026-04-06,2026-04-10,"1,270.20",5.58
154,Nifty 50,Largest loss,2024-12-16,2024-12-20,"-1,122.05",-4.53
468,Sensex,Largest gain,2026-04-06,2026-04-10,"4,059.76",5.53
400,Sensex,Largest loss,2024-12-16,2024-12-20,"-3,788.82",-4.62


,instrument,year,weeks,up_weeks_pct,mean_move_pct,mean_abs_move_pct,p90_abs_move_pct
0,Nifty 50,2022,48.00,54.17,0.29,1.69,3.09
1,Nifty 50,2023,46.00,50.00,0.07,1.13,2.32
2,Nifty 50,2024,46.00,56.52,0.02,1.22,2.28
3,Nifty 50,2025,48.00,47.92,0.21,1.26,2.55
4,Nifty 50,2026,32.00,43.75,-0.10,1.44,2.87
5,Sensex,2022,47.00,53.19,0.31,1.77,3.14
6,Sensex,2023,46.00,52.17,0.12,1.16,2.44
7,Sensex,2024,46.00,58.70,0.06,1.21,2.35
8,Sensex,2025,48.00,47.92,0.22,1.25,2.38
9,Sensex,2026,32.00,43.75,-0.13,1.54,3.10


## Compare returns and intraweek movement

Hover for the exact values and zoom to a date range. The first chart compares
matching weeks. The second shows each index's observed high/low moves from
Monday's open alongside its Friday return: a small final return can still
contain a large move during the week.

In [6]:
comparison = weekly.pivot(index="week_start", columns="instrument", values="endpoint_move_pct")
comparison = comparison.reindex(columns=list(SOURCES)).dropna()
fig = go.Figure()
for instrument in SOURCES:
    fig.add_trace(go.Scatter(
        x=comparison.index, y=comparison[instrument], name=instrument, mode="lines",
        line=dict(color=COLORS[instrument], width=1.6),
        hovertemplate="Week of %{x|%Y-%m-%d}<br>Move: %{y:+.2f}%<extra>%{fullData.name}</extra>",
    ))
fig.add_hline(y=0, line_color="#999", line_width=1)
fig.update_layout(
    title=f"Monday open → Friday close · {len(comparison)} common weeks",
    height=440, hovermode="x unified", yaxis_title="Move (%)",
    xaxis=dict(title="Week starting Monday", rangeslider=dict(visible=True)),
    legend=dict(orientation="h", y=1.1),
)
fig.show()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=list(SOURCES))
for row, instrument in enumerate(SOURCES, start=1):
    group = weekly.loc[weekly["instrument"].eq(instrument)].sort_values("week_start")
    for column, label, color in [
        ("max_up_pct", "High vs Monday open", "#1baf7a"),
        ("max_down_pct", "Low vs Monday open", "#d75555"),
        ("endpoint_move_pct", "Friday return", "#2a78d6"),
    ]:
        fig.add_trace(go.Scatter(
            x=group["week_start"], y=group[column], name=label, legendgroup=label,
            showlegend=row == 1, mode="lines", line=dict(color=color, width=1.2),
            hovertemplate="%{x|%Y-%m-%d}: %{y:+.2f}%<extra>%{fullData.name}</extra>",
        ), row=row, col=1)
    fig.add_hline(y=0, line_color="#aaa", line_width=1, row=row, col=1)
fig.update_layout(title="Weekly high, low and Friday return from Monday open", height=650, hovermode="x unified")
fig.update_yaxes(title_text="Move (%)")
fig.update_xaxes(title_text="Week starting Monday", row=2, col=1)
fig.show()

## How often does the weekly move stay within a band?

`close_inside_pct` counts Friday returns within ± the stated percentage of Monday's
open. `observed_range_inside_pct` requires **every supplied hourly high and low**
to stay inside that same band. These are historical index-move frequencies,
not option P&L calculations. Boundary values count as inside.

In [7]:
band_rows = []
for instrument, group in weekly.groupby("instrument"):
    for distance in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
        close_inside = group["endpoint_move_pct"].abs().le(distance)
        range_inside = group["max_up_pct"].le(distance) & group["max_down_pct"].ge(-distance)
        band_rows.append({
            "instrument": instrument, "band_pct": distance, "weeks": len(group),
            "close_inside_weeks": int(close_inside.sum()),
            "close_inside_pct": close_inside.mean() * 100,
            "observed_range_inside_weeks": int(range_inside.sum()),
            "observed_range_inside_pct": range_inside.mean() * 100,
        })
bands = pd.DataFrame(band_rows)
display(bands)

fig = go.Figure()
for instrument, group in weekly.groupby("instrument"):
    fig.add_trace(go.Histogram(
        x=group["endpoint_move_pct"], name=f"{instrument} ({len(group)} weeks)",
        histnorm="percent", xbins=dict(start=-12, end=12, size=0.5),
        marker_color=COLORS[instrument], opacity=0.65,
    ))
fig.update_layout(
    title="Distribution of Monday-to-Friday returns", barmode="overlay",
    xaxis_title="Move (%)", yaxis_title="Share of included weeks (%)", height=400,
)
fig.show()

,instrument,band_pct,weeks,close_inside_weeks,close_inside_pct,observed_range_inside_weeks,observed_range_inside_pct
0,Nifty 50,0.50,220,48,21.82,0,0.00
1,Nifty 50,1.00,220,104,47.27,31,14.09
2,Nifty 50,1.50,220,141,64.09,80,36.36
3,Nifty 50,2.00,220,168,76.36,126,57.27
4,Nifty 50,2.50,220,189,85.91,165,75.00
5,Nifty 50,3.00,220,207,94.09,186,84.55
6,Nifty 50,4.00,220,215,97.73,208,94.55
7,Nifty 50,5.00,220,219,99.55,214,97.27
8,Sensex,0.50,219,49,22.37,1,0.46
9,Sensex,1.00,219,102,46.58,29,13.24


## Export all results

CSV files are saved under `data/processed/option_selling/weekly_monday_friday/`.
`all_weeks_audit.csv` includes excluded weeks; its price statistics may cover only
part of the week, so use `included_weeks.csv` for the analyzed sample. All CSV
timestamps without an offset represent Asia/Kolkata local time. Rerunning this
cell replaces these generated files using the current parameters.

In [8]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    "source_coverage.csv": coverage.reset_index(),
    "all_weeks_audit.csv": weekly_all,
    "included_weeks.csv": weekly,
    "summary.csv": summary.reset_index(),
    "annual_summary.csv": annual,
    "move_bands.csv": bands,
    "common_week_returns.csv": comparison.reset_index(),
}
for filename, frame in exports.items():
    frame.to_csv(EXPORT_DIR / filename, index=False)
for instrument, stem in [("Nifty 50", "nifty50"), ("Sensex", "sensex")]:
    weekly.loc[weekly["instrument"].eq(instrument)].to_csv(EXPORT_DIR / f"{stem}_weekly_moves.csv", index=False)
print(f"Saved {len(exports) + len(SOURCES)} CSV files to {EXPORT_DIR.relative_to(PROJECT_ROOT)}")

Saved 9 CSV files to data/processed/option_selling/weekly_monday_friday
